#Data Mining and Analysis Final Project
The purpose of this project is to analyze a social media dataset with the intent of discovering possible associative behavior between different content types, social media platforms and topics of social media posts.

#**Title: Social Media Topics for You!!!**

#What are we Analyzing?

The focuse of this project will be a dataset extracted from the datascience platform Kaggle. The dataset, made by user "S.V.Thejaswini," is comprised of various content types and audience engagement metrics used in social media. To be more precise, the columns of data are:

- Platform (name of the social media platform where the content was published)
- Post_ID (unique identifier assigned to each social media post for tracking and analysis)
- Post_Type (type or format of the post such as Image, Video, Carousel, Reel, or Text)
- Post_Date (Date on which the post was published on the platform)
- Likes (the total number of likes received by the post, indicating user appreciation)
- Comments (number of comments generated by the post, reflecting audience interaction)
- Shares (number of times the post was shared, showing content virality)
- Reach (total number of unique users who viewed the post)
- Impressions (total number of times the post was displayed, including repeated views)
- Engagement_Rate (percentric metric representing representing overall engagement based on likes, comments, and shares relative to reach or impressions)
- Clicks (number of user clicks on the post {links, profile visits, or call-to-action interactions})
- Follower_Count (total number of followers on the platform at the time the post was published)
- Hashtags_Used (number of hashtags included in the post to improve discoverability)
- Content_Length (length of the post caption or text, measured in characters or words)
- Posting_Time (time of day when the post was published, useful for timing analysis)

# Research Question

What is the association between a social media post's content type and topic? Say we have a video post, what topic would most likely be associated with it, lifestyle, food, health? In addition to this, what if we take a post social media platform and content type, what is the association between that pair and the post's topic? If the post was a reel from Instagram, what topic would it most likely have?

#Why do this?

The analysis from this project can help social media platforms determine what topics to recommend to their content creators what topics to dive into depending on the format of their content posts. Platforms who are privy to a certain content type can also recommend to their users posts relating to the topics most associated with that content type. For instance, since YouTube is predominantly a video service and if it happens that travel is the most associated topic with YouTube videos, then the platform can recommend travel videos to users who have just joined and don't yet know what to watch.

#Part 1: Time to Explore!

I want to explore the idea of topic and content type associations from the ground up using just basic association rule mining and evaluating them utilizinf confidence, interest, and lift. This part of the analysis will just find the associations between single element itemsets: a post's content type and the post's topic.

In [12]:
from google.colab import files
from google.colab import drive
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

!git clone https://github.com/connorh3333/CSCE-676-Project---Social-Media-Performance-and-Engagement-Analysis.git


df = pd.read_csv("CSCE-676-Project---Social-Media-Performance-and-Engagement-Analysis/data/social_media_performance.csv")

fatal: destination path 'CSCE-676-Project---Social-Media-Performance-and-Engagement-Analysis' already exists and is not an empty directory.


In [13]:
from IPython.display import display, Markdown, HTML

df.head()

,post_id,platform,content_type,topic,language,region,post_datetime,hashtags,sentiment_score,views,likes,comments,shares,engagement_rate,is_viral
0,1,LinkedIn,article,Technology,UR,BR,2025-04-25 09:47:00,#AI #Innovation #TechTrends #Programming #Codi...,0.76,37781,1202,462,185,0.0490,0
1,2,LinkedIn,poll,Health,FR,JP,2025-10-29 09:44:00,#Fitness #Nutrition #Wellness #Health #MentalH...,0.46,23541,1399,538,215,0.0915,1
2,3,LinkedIn,article,Travel,HI,FR,2025-02-10 14:12:00,#Travel #Journey #Adventure #Tourism #ExploreM...,-0.01,30714,1663,639,255,0.0833,1
3,4,LinkedIn,image,Sports,DE,DE,2025-04-18 22:41:00,#Cricket #Workout #Fitness,0.55,31294,1372,528,211,0.0675,0
4,5,LinkedIn,poll,Business,DE,US,2025-04-28 10:17:00,#Entrepreneur #Leadership #StartupLife,0.70,43129,2234,859,343,0.0797,0


#Type-->Topic or Topic-->Type?

A difficulty in this part of the analysis is realizinf that confidence and interest are not bidirectional: conf(I-->J) does not equal conf(J-->I). So then comes the question, which direction should we choose? content type --> topic or topic --> content type? On one hand, conf(type --> topic) tells us that "given a this content type, what topic is most associated with it?" On the other side of the coin, conf(topic --> type) tells us "given a post's topic, what content type is most associated with it?" These may seem like the are answering the same question, but in reality conf(topic --> type) and conf(type --> topic) can give completely different values. The differences in these two directions made me curious so I wanted to explore both. In order to accomplish this, I constructed support, confidence, and interest functions that can handle either direction.

In [14]:
def supp(itemset, df):

  if not itemset:
    return 1.0

  mask = pd.Series(True, index=df.index)

  for col, val in itemset:

    current_condition = df[col] == val

    mask &= current_condition

  return mask.mean()





In [15]:
def conf(I, J, df):

  supp_I = supp(I, df)

  if supp_I == 0:
    return 0.0

  supp_I_J = supp(I + J, df)

  confidence = supp_I_J / supp_I

  return confidence

In [16]:
def interest(I, J, df):

  return abs(conf(I, J, df) - supp(J, df))

Since lift actually is bidirectional, there is no need to pay attention what feature is added in first: topic or content type.

In [17]:
def lift(I, J, df):

  supp_J = supp(J, df)

  if supp_J == 0:
    return 0

  return conf(I, J, df) / supp_J

In [18]:
type_list = list(set(df["content_type"]))
topic_list = list(set(df["topic"]))

table_data_top_typ = []
table_data_typ_top = []
table_data_interest_typ_top = []
table_data_interest_top_typ = []
table_data_lift = []

for typ in type_list:
  row_top_typ = []
  row_typ_top = []
  row_interest_typ_top = []
  row_interest_top_typ = []
  row_lift = []
  for top in topic_list:

    I_type = [("content_type", typ)]
    I_topic = [("topic", top)]

    topic_to_type = conf(I_topic, I_type, df)
    type_to_topic = conf(I_type, I_topic, df)
    interest_score_typ_top = interest(I_type, I_topic, df)
    interest_score_top_typ = interest(I_topic, I_type, df)
    lift_score = lift(I_topic, I_type, df)

    row_top_typ.append(topic_to_type)
    row_typ_top.append(type_to_topic)
    row_interest_typ_top.append(interest_score_typ_top)
    row_interest_top_typ.append(interest_score_top_typ)
    row_lift.append(lift_score)

  table_data_top_typ.append(row_top_typ)
  table_data_typ_top.append(row_typ_top)
  table_data_interest_typ_top.append(row_interest_typ_top)
  table_data_interest_top_typ.append(row_interest_top_typ)
  table_data_lift.append(row_lift)

topic_type_table = pd.DataFrame(table_data_top_typ, index=type_list, columns=topic_list)
type_topic_table = pd.DataFrame(table_data_typ_top, index=type_list, columns=topic_list)
interest_table_typ_top = pd.DataFrame(table_data_interest_typ_top, index=type_list, columns=topic_list)
interest_table_top_typ = pd.DataFrame(table_data_interest_top_typ, index=type_list, columns=topic_list)
lift_table = pd.DataFrame(table_data_lift, index=type_list, columns=topic_list)

topic_type_table.index.name = "content_type"
topic_type_table.columns.name = "topic"

type_topic_table.index.name = "content_type"
type_topic_table.columns.name = "topic"

interest_table_typ_top.index.name = "content_type"
interest_table_typ_top.columns.name = "topic"

interest_table_top_typ.index.name = "content_type"
interest_table_top_typ.columns.name = "topic"

lift_table.index.name = "content_type"
lift_table.columns.name = "topic"

display(Markdown("##  CONFIDENCE TABLE (TOPIC --> CONTENT TYPE):"))
display(topic_type_table)

display(HTML("<br><br><hr><br>"))

display(Markdown("##  CONFIDENCE TABLE (CONTENT TYPE --> TOPIC):"))
display(type_topic_table)

display(HTML("<br><br><hr><br>"))

display(Markdown("##  INTEREST TABLE (CONTENT TYPE --> TOPIC):"))
display(interest_table_typ_top)

display(HTML("<br><br><hr><br>"))

display(Markdown("##  INTEREST TABLE (TOPIC --> CONTENT TYPE):"))
display(interest_table_top_typ)

display(HTML("<br><br><hr><br>"))

display(Markdown("##  LIFT TABLE:"))
display(lift_table)

##  CONFIDENCE TABLE (TOPIC --> CONTENT TYPE):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
content_type,,,,,,,,,,
video,0.304603,0.309709,0.311220,0.308577,0.276699,0.310030,0.290675,0.308628,0.299197,0.289549
image,0.107738,0.097087,0.101463,0.097280,0.098058,0.106383,0.092262,0.099558,0.093373,0.097795
poll,0.066601,0.051456,0.049756,0.051255,0.044660,0.038501,0.056548,0.053097,0.055221,0.057526
article,0.144956,0.166990,0.189268,0.164226,0.200000,0.165147,0.177579,0.184735,0.176707,0.173538
reel,0.054848,0.056311,0.047805,0.050209,0.038835,0.051672,0.055556,0.050885,0.047189,0.054650
feed,0.041136,0.054369,0.044878,0.050209,0.052427,0.051672,0.044643,0.060841,0.048193,0.047939
carousel,0.101861,0.096117,0.084878,0.100418,0.105825,0.096251,0.086310,0.078540,0.101406,0.091083
story,0.178257,0.167961,0.170732,0.177824,0.183495,0.180344,0.196429,0.163717,0.178715,0.187919


##  CONFIDENCE TABLE (CONTENT TYPE --> TOPIC):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
content_type,,,,,,,,,,
video,0.103425,0.106086,0.106086,0.098104,0.094779,0.101763,0.097439,0.092784,0.099102,0.100432
image,0.110999,0.100908,0.104945,0.093845,0.101917,0.105954,0.093845,0.090817,0.093845,0.102926
poll,0.129524,0.100952,0.097143,0.093333,0.087619,0.072381,0.108571,0.091429,0.104762,0.114286
article,0.084911,0.098680,0.111302,0.090075,0.118187,0.093517,0.102697,0.095812,0.100975,0.103844
reel,0.110236,0.114173,0.096457,0.094488,0.078740,0.100394,0.110236,0.090551,0.092520,0.112205
feed,0.084848,0.113131,0.092929,0.096970,0.109091,0.103030,0.090909,0.111111,0.096970,0.101010
carousel,0.110169,0.104873,0.092161,0.101695,0.115466,0.100636,0.092161,0.075212,0.106992,0.100636
story,0.101847,0.096810,0.097929,0.095132,0.105764,0.099608,0.110800,0.082820,0.099608,0.109681


##  INTEREST TABLE (CONTENT TYPE --> TOPIC):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
content_type,,,,,,,,,,
video,0.001325,0.003086,0.003586,0.002504,0.008221,0.003063,0.003361,0.002384,0.000498,0.003868
image,0.008899,0.002092,0.002445,0.001755,0.001083,0.007254,0.006955,0.000417,0.005755,0.001374
poll,0.027424,0.002048,0.005357,0.002267,0.015381,0.026319,0.007771,0.001029,0.005162,0.009986
article,0.017189,0.004320,0.008802,0.005525,0.015187,0.005183,0.001897,0.005412,0.001375,0.000456
reel,0.008136,0.011173,0.006043,0.001112,0.024260,0.001694,0.009436,0.000151,0.007080,0.007905
feed,0.017252,0.010131,0.009571,0.001370,0.006091,0.004330,0.009891,0.020711,0.002630,0.003290
carousel,0.008069,0.001873,0.010339,0.006095,0.012466,0.001936,0.008639,0.015188,0.007392,0.003664
story,0.000253,0.006190,0.004571,0.000468,0.002764,0.000908,0.010000,0.007580,0.000008,0.005381


##  INTEREST TABLE (TOPIC --> CONTENT TYPE):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
content_type,,,,,,,,,,
video,0.003903,0.009009,0.010520,0.007877,0.024001,0.009330,0.010025,0.007928,0.001503,0.011151
image,0.008638,0.002013,0.002363,0.001820,0.001042,0.007283,0.006838,0.000458,0.005727,0.001305
poll,0.014101,0.001044,0.002744,0.001245,0.007840,0.013999,0.004048,0.000597,0.002721,0.005026
article,0.029344,0.007310,0.014968,0.010074,0.025700,0.009153,0.003279,0.010435,0.002407,0.000762
reel,0.004048,0.005511,0.002995,0.000591,0.011965,0.000872,0.004756,0.000085,0.003611,0.003850
feed,0.008364,0.004869,0.004622,0.000709,0.002927,0.002172,0.004857,0.011341,0.001307,0.001561
carousel,0.007461,0.001717,0.009522,0.006018,0.011425,0.001851,0.008090,0.015860,0.007006,0.003317
story,0.000443,0.010739,0.007968,0.000876,0.004795,0.001644,0.017729,0.014983,0.000015,0.009219


##  LIFT TABLE:

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
content_type,,,,,,,,,,
video,1.012981,1.029959,1.034983,1.026197,0.920183,1.031029,0.966660,1.026366,0.995001,0.962918
image,1.087160,0.979691,1.023849,0.981638,0.989488,1.073491,0.930998,1.004617,0.942215,0.986830
poll,1.268598,0.980120,0.947735,0.976290,0.850670,0.733343,1.077098,1.011378,1.051826,1.095740
article,0.831646,0.958062,1.085877,0.942203,1.147447,0.947487,1.018814,1.059865,1.013809,0.995627
reel,1.079689,1.108478,0.941041,0.988370,0.764468,1.017160,1.093613,1.001672,0.928913,1.075788
feed,0.831033,1.098362,0.906627,1.014327,1.059135,1.043873,0.901876,1.229105,0.973591,0.968457
carousel,1.079035,1.018183,0.899132,1.063754,1.121030,1.019611,0.914296,0.831990,1.074212,0.964867
story,0.997519,0.939906,0.955410,0.995099,1.026833,1.009202,1.099209,0.916155,1.000083,1.051592


The confidence tables are both very different as the conf(type-->topic) table is evenly distributed while the conf(topic-->type) table is mostly concentrated on the "video" content type. This proves that an association rule's confidence is very sensitive to the direction of it's itemsets.

Each confidence table has it's own specific uses. conf(topic-->type) can be used for content creators that already know the topic that they want to talk about but don't know what format to make their post. conf(type-->topic) can be used to help content creators who have chosen a specific format but do not know what topic to dive into.

Since we are mostly focusing on recommending content creators topics, we will focus more on the latter. As mentioned before, the distribution for conf(type-->topic) is pretty even with relatively low confidence scores overall. The highest amongst them are associations like carousel --> fashion and feed --> business.

All of this suggests that there might not be strong associations between a post's type and it's topic.

The interest actually tells us how meaningful the confidences are by telling us how commanplace one itemset is amongst the other. Since all of the interest scores, for both interest(type-->topic) and interest(topic-->type), are very low we can deduce that the none of the confidence scores are very surprising.

However, since confidence can often be misleading, I wanted to use lift as my final metric.

Luckily, since lift is bidirectional, we do not need to account for the direction of the inputs for lift.

The lift table indicates that the vast majority of topics and content types have little to no association at all as most of their scores are approximately 1.

An interesting thing to note though, after looking at both confidence tabels, one would assume that there would be a positive association between "video" and "entertainment" since that combination received relative high scores in both tables. However, the lift table shows that "video" and "entertainment" have almost no relationship at all. This showcases how misleading and unreliable confidence can be.

#Part 2: Let's Change the Recipe

Now let's make the analysis more optimal for social media platforms in particular by finding associations between {post's platform, post's content type} and {post's topic}. This in particular can help a platform know which content type, topic combination is best suited for them (a reel-->food association might work well for Instagram but not YouTube).

In order to accomplish this, I am going to use Apriori to find the most frequent itemsets and then utilized our prestablished metrics for evaluating the associations of the most frequent itemsets.

An important thing to note, since we are focusing on the direction {post's platform, post's content type}-->{post's topic} we will only need one confidence table.

In [19]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
import warnings

warnings.simplefilter("ignore", category=DeprecationWarning)

transactions = []

for idx, row in df.iterrows():
  tran_row = []

  platform_item = "platform=" + row["platform"]
  type_item = "content_type=" + row["content_type"]
  topic_item = "topic=" + row["topic"]

  tran_row.append(platform_item)
  tran_row.append(type_item)
  tran_row.append(topic_item)

  transactions.append(tran_row)

tran_enc = TransactionEncoder()

encoded_trans = tran_enc.fit(transactions).transform(transactions)

trans_df = pd.DataFrame(encoded_trans, columns=tran_enc.columns_)

min_supp = 0.05


freq_items = apriori(trans_df, min_support=min_supp, use_colnames=True)

target_items = []

for idx, row in freq_items.iterrows():
  itemset = row["itemsets"]

  if len(itemset) == 2 and any("platform=" in s for s in itemset) and any("content_type=" in s for s in itemset):
    target_items.append(itemset)


apriori_conf_data = []
apriori_lift_data = []

for itemset in target_items:

  apriori_conf_row = []
  apriori_lift_row = []

  for top in topic_list:

    I_set = []

    for item in itemset:
      parts = item.split("=")

      I_set.append((parts[0], parts[1]))

    I_topic = [("topic", top)]

    conf_val = conf(I_set, I_topic, df)
    lift_val = lift(I_set, I_topic, df)

    apriori_conf_row.append(conf_val)
    apriori_lift_row.append(lift_val)

  apriori_conf_data.append(apriori_conf_row)
  apriori_lift_data.append(apriori_lift_row)

apriori_conf_table = pd.DataFrame(apriori_conf_data, index=target_items, columns=topic_list)
apriori_lift_table = pd.DataFrame(apriori_lift_data, index=target_items, columns=topic_list)

apriori_conf_table.index.name = "platform + content_type"
apriori_conf_table.columns.name = "topic"

apriori_lift_table.index.name = "platform + content_type"
apriori_lift_table.columns.name = "topic"

display(HTML("<br><br><hr><br>"))
display(Markdown("## APRIORI CONFIDENCE TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):"))
display(apriori_conf_table)

display(HTML("<br><br><hr><br>"))
display(Markdown("## APRIORI LIFT TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):"))
display(apriori_lift_table)


## APRIORI CONFIDENCE TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
platform + content_type,,,,,,,,,,
"(platform=LinkedIn, content_type=article)",0.102970,0.100990,0.104950,0.087129,0.118812,0.108911,0.089109,0.083168,0.106931,0.097030
"(platform=Medium, content_type=article)",0.077544,0.097738,0.113893,0.091276,0.117932,0.087237,0.108239,0.100969,0.098546,0.106624
"(content_type=image, platform=Instagram)",0.115768,0.117764,0.109780,0.081836,0.095808,0.109780,0.105788,0.081836,0.087824,0.093812
"(content_type=poll, platform=LinkedIn)",0.129524,0.100952,0.097143,0.093333,0.087619,0.072381,0.108571,0.091429,0.104762,0.114286
"(platform=Instagram, content_type=reel)",0.110236,0.114173,0.096457,0.094488,0.078740,0.100394,0.110236,0.090551,0.092520,0.112205
"(platform=Instagram, content_type=story)",0.087619,0.097143,0.100952,0.102857,0.099048,0.093333,0.125714,0.076190,0.097143,0.120000
"(platform=Medium, content_type=story)",0.107765,0.096672,0.096672,0.091918,0.108558,0.102219,0.104596,0.085578,0.100634,0.105388
"(content_type=video, platform=LinkedIn)",0.120316,0.120316,0.088757,0.088757,0.094675,0.104536,0.078895,0.102564,0.096647,0.104536
"(content_type=video, platform=YouTube)",0.100000,0.103200,0.109600,0.100000,0.094800,0.101200,0.101200,0.090800,0.099600,0.099600


## APRIORI LIFT TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
platform + content_type,,,,,,,,,,
"(platform=LinkedIn, content_type=article)",1.008524,0.980486,1.023907,0.911388,1.153513,1.103454,0.884017,0.920004,1.073601,0.930294
"(platform=Medium, content_type=article)",0.759495,0.948915,1.111155,0.954773,1.144972,0.883865,1.073801,1.116917,0.989418,1.022278
"(content_type=image, platform=Instagram)",1.133873,1.143344,1.071029,0.856029,0.930178,1.112264,1.049488,0.905269,0.881771,0.899448
"(content_type=poll, platform=LinkedIn)",1.268598,0.980120,0.947735,0.976290,0.850670,0.733343,1.077098,1.011378,1.051826,1.095740
"(platform=Instagram, content_type=reel)",1.079689,1.108478,0.941041,0.988370,0.764468,1.017160,1.093613,1.001672,0.928913,1.075788
"(platform=Instagram, content_type=story)",0.858169,0.943135,0.984901,1.075912,0.961627,0.945626,1.247166,0.842815,0.975330,1.150527
"(platform=Medium, content_type=story)",1.055489,0.938563,0.943141,0.961481,1.053960,1.035650,1.037658,0.946664,1.010381,1.010434
"(content_type=video, platform=LinkedIn)",1.178409,1.168112,0.865926,0.928425,0.919170,1.059134,0.782693,1.134559,0.970351,1.002267
"(content_type=video, platform=YouTube)",0.979432,1.001942,1.069268,1.046025,0.920388,1.025329,1.003968,1.004425,1.000000,0.954938


Like Part 1, the confidence table shows very small scores indicating a lack of association amongst the content types and topics.

However, as also established in Part 1, lift is the better metric. While a large number of associations in the table are weak, there are some stand out rules. LinkedIn polls and health have a very strong association and LinkedIn videos and health and food also have positive associations.

One association rule that I was very surprised by was the one between YouTube videos and the topic of technology. Many people I know get their information about technology from watching videos on YouTube so I would assume there would be a positive association. However, the lift table says otherwise as it shows that YouTube videos and the topic of technology are completely independent of each other with a lift score of a surprisingly exact 1.000.

#Part 3: Fine Tuning Time

In this part of the analysis I want to perform the same tasks as Part 2 but this time incorporate SON instead of just Apriori alone. The reason why is to increase computation efficiency and to help it scale to larger datasets.

In [20]:
import math

part_size = math.ceil(len(transactions) / 4)

partitions = []

for i in range(0, len(transactions), part_size):
  partition = transactions[i:i + part_size]
  partitions.append(partition)

candidate_itemsets = set()

for partition in partitions:

  encoded_part = tran_enc.fit(partition).transform(partition)

  part_df = pd.DataFrame(encoded_part, columns=tran_enc.columns_)

  min_supp_part = min_supp * (len(partition)/len(transactions))

  freq_part = apriori(part_df, min_support=min_supp_part, use_colnames=True)

  for itemset in freq_part["itemsets"]:

        candidate_itemsets.add(itemset)

verified_itemsets = []
verified_supports = []

for candidate in candidate_itemsets:

  count = 0

  for transaction in transactions:
    if candidate.issubset(set(transaction)):
      count += 1

  global_support = count / len(transactions)

  if global_support >= min_supp:
    verified_itemsets.append(candidate)
    verified_supports.append(global_support)


son_freq_items = pd.DataFrame({"support":verified_supports, "itemsets":verified_itemsets})


son_target_items = []

for idx, row in son_freq_items.iterrows():
  itemset = row["itemsets"]

  if len(itemset) == 2 and any("platform=" in s for s in itemset) and any("content_type=" in s for s in itemset):
    son_target_items.append(itemset)


son_conf_data = []
son_lift_data = []

for itemset in son_target_items:

  son_conf_row = []
  son_lift_row = []

  for top in topic_list:

    I_set = []

    for item in itemset:
      parts = item.split("=")

      I_set.append((parts[0], parts[1]))

    I_topic = [("topic", top)]

    conf_val = conf(I_set, I_topic, df)
    lift_val = lift(I_set, I_topic, df)

    son_conf_row.append(conf_val)
    son_lift_row.append(lift_val)

  son_conf_data.append(son_conf_row)
  son_lift_data.append(son_lift_row)

son_conf_table = pd.DataFrame(son_conf_data, index=son_target_items, columns=topic_list)
son_lift_table = pd.DataFrame(son_lift_data, index=son_target_items, columns=topic_list)

son_conf_table.index.name = "platform + content_type"
son_conf_table.columns.name = "topic"

son_lift_table.index.name = "platform + content_type"
son_lift_table.columns.name = "topic"

display(HTML("<br><br><hr><br>"))
display(Markdown("## SON CONFIDENCE TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):"))
display(son_conf_table)

display(HTML("<br><br><hr><br>"))
display(Markdown("## SON LIFT TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):"))
display(son_lift_table)





## SON CONFIDENCE TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
platform + content_type,,,,,,,,,,
"(platform=LinkedIn, content_type=article)",0.102970,0.100990,0.104950,0.087129,0.118812,0.108911,0.089109,0.083168,0.106931,0.097030
"(content_type=image, platform=Instagram)",0.115768,0.117764,0.109780,0.081836,0.095808,0.109780,0.105788,0.081836,0.087824,0.093812
"(content_type=video, platform=LinkedIn)",0.120316,0.120316,0.088757,0.088757,0.094675,0.104536,0.078895,0.102564,0.096647,0.104536
"(content_type=video, platform=YouTube)",0.100000,0.103200,0.109600,0.100000,0.094800,0.101200,0.101200,0.090800,0.099600,0.099600
"(content_type=story, platform=Instagram)",0.087619,0.097143,0.100952,0.102857,0.099048,0.093333,0.125714,0.076190,0.097143,0.120000
"(content_type=poll, platform=LinkedIn)",0.129524,0.100952,0.097143,0.093333,0.087619,0.072381,0.108571,0.091429,0.104762,0.114286
"(platform=Instagram, content_type=reel)",0.110236,0.114173,0.096457,0.094488,0.078740,0.100394,0.110236,0.090551,0.092520,0.112205
"(platform=Medium, content_type=story)",0.107765,0.096672,0.096672,0.091918,0.108558,0.102219,0.104596,0.085578,0.100634,0.105388
"(platform=Medium, content_type=article)",0.077544,0.097738,0.113893,0.091276,0.117932,0.087237,0.108239,0.100969,0.098546,0.106624


## SON LIFT TABLE ((PLATFORM, CONTENT TYPE) --> TOPIC):

topic,Health,Food,Fashion,Education,Travel,Entertainment,Sports,Business,Technology,Lifestyle
platform + content_type,,,,,,,,,,
"(platform=LinkedIn, content_type=article)",1.008524,0.980486,1.023907,0.911388,1.153513,1.103454,0.884017,0.920004,1.073601,0.930294
"(content_type=image, platform=Instagram)",1.133873,1.143344,1.071029,0.856029,0.930178,1.112264,1.049488,0.905269,0.881771,0.899448
"(content_type=video, platform=LinkedIn)",1.178409,1.168112,0.865926,0.928425,0.919170,1.059134,0.782693,1.134559,0.970351,1.002267
"(content_type=video, platform=YouTube)",0.979432,1.001942,1.069268,1.046025,0.920388,1.025329,1.003968,1.004425,1.000000,0.954938
"(content_type=story, platform=Instagram)",0.858169,0.943135,0.984901,1.075912,0.961627,0.945626,1.247166,0.842815,0.975330,1.150527
"(content_type=poll, platform=LinkedIn)",1.268598,0.980120,0.947735,0.976290,0.850670,0.733343,1.077098,1.011378,1.051826,1.095740
"(platform=Instagram, content_type=reel)",1.079689,1.108478,0.941041,0.988370,0.764468,1.017160,1.093613,1.001672,0.928913,1.075788
"(platform=Medium, content_type=story)",1.055489,0.938563,0.943141,0.961481,1.053960,1.035650,1.037658,0.946664,1.010381,1.010434
"(platform=Medium, content_type=article)",0.759495,0.948915,1.111155,0.954773,1.144972,0.883865,1.073801,1.116917,0.989418,1.022278


Fortunetly, the results of the SON implementation are the same as those of the normal Apriori implementation meaning that this code for SON is correct.

An important thought: this code was hard coded for 4 partions in particular, so if we wanted to make the SON code more adaptable to increasingly large datasets, we would need to alter the code so that the number of partitions would change depending on either the user or the size of the dataset.

#Takeaways



*   When analyzing the content types and topics on their own, most of them did not possess any strong associations
*   When changing the itemsets to {platform, type} --> {topic} more strong associations appeared in comparison to the single item itemset analysis but the majority of associations were weak



#Conclusion

The majority of the analysis suggests that there is not really a strong association between a post's content type and the post's topic. While this is dissappointing, there were some noticeable associations: {Instagram, image}-->{health}, {Medium, article}-->{fashion}, {LinkedIn, video}-->{food}, and more. I believe that these associations, however small, can still help to optimize the social media experience for both content viewers and content creators alike.

#Difficulties


*   The first difficulty I had was deciding on the confidence/interest direction for content type and topic and deciding which would be more applicable to a real-world setting
*   I also struggled with implementing the SON algorithm as there is no Python library that I know of that can readily use it, so I had to implement it from scratch.



#Limitations


*   This dataset is comprised of data from just one year so it is possible that the sample size of the dataset might be too small.



In [21]:
!pip freeze > requirements.txt
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!python --version

Python 3.12.13
